In [ ]:
from pathlib import Path

# Run this baseline for the attached SemanticCloneBench V3 Kaggle dataset.
# Add the SemanticCloneBench V3 dataset to the Kaggle notebook inputs before running all cells.
DATASET_KEYS = ("semantic-clone-bench",)
RUN_LABEL = "cross_language_astnn"


# Kaggle's current PyTorch build cannot execute kernels on Tesla P100 (sm_60).
# These notebooks are intended for a T4-class accelerator; two T4s are fine,
# although this single-process implementation uses GPU 0.
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU is enabled. In Kaggle select GPU accelerator: 2x T4.")
_GPU_CAPABILITY = torch.cuda.get_device_capability(0)
_GPU_NAME = torch.cuda.get_device_name(0)
if _GPU_CAPABILITY[0] < 7:
    raise RuntimeError(
        f"{_GPU_NAME} has unsupported CUDA capability sm_{_GPU_CAPABILITY[0]}{_GPU_CAPABILITY[1]}. "
        "Select 2x T4 in Kaggle Accelerator settings, restart the session, and Run All."
    )
print({"gpu": _GPU_NAME, "capability": f"sm_{_GPU_CAPABILITY[0]}{_GPU_CAPABILITY[1]}", "gpu_count": torch.cuda.device_count()})
# Runtime profile. Use quick_1h for preliminary results; change only this
# value to extended_6_7h for the larger follow-up run.
RUN_PROFILE = "comparison_50k"
RUN_PRESETS = {"quick_1h": {'max_train_pairs': 50000, 'max_valid_pairs': 10000, 'max_test_pairs': 10000, 'epochs': 8, 'patience': 2}, "extended_6_7h": {'max_train_pairs': 100000, 'max_valid_pairs': 20000, 'max_test_pairs': 20000, 'epochs': 30, 'patience': 6}}
# Use this profile in every method notebook for a data-equal comparison.
RUN_PRESETS["comparison_50k"] = {
    **RUN_PRESETS["quick_1h"],
    "max_train_pairs": 50_000,
    "max_valid_pairs": 10_000,
    "max_test_pairs": 10_000,
}

# Final paper protocol: use every available pair in each official split.
RUN_PRESETS["final_full"] = {
    **RUN_PRESETS["quick_1h"],
    "max_train_pairs": None,
    "max_valid_pairs": None,
    "max_test_pairs": None,
}

# --- bounded run budget ---
# Kaggle sessions are capped, and a run that dies at the limit produces nothing.
# Training data stays large so results remain comparable with the published
# table; validation and test are capped because a bigger validation split only
# sharpens one threshold, and a bigger test split only tightens an error bar we
# do not report.
RUN_PRESETS["bounded_10h"] = {
    **RUN_PRESETS["comparison_50k"],
    "max_train_pairs": 200_000,
    "max_valid_pairs": 20_000,
    "max_test_pairs": 20_000,
}

if RUN_PROFILE not in RUN_PRESETS:
    raise ValueError(f"Unknown RUN_PROFILE={RUN_PROFILE!r}; choose one of {tuple(RUN_PRESETS)}")
RUN_CONFIG = RUN_PRESETS[RUN_PROFILE]


In [ ]:
# === per-language breakdown helper ===
# Splits an already-computed set of test predictions by the language of each
# pair. No retraining and no separate per-language model: this is the same run,
# reported per language so a strong average cannot hide a collapsed language.
import gzip as _gzip
import json as _json
from pathlib import Path as _Path

import numpy as _np
import pandas as _pd

_LANGUAGE_CACHE = {}
LANGUAGE_BREAKDOWN_ROWS = []


def _resolve_codes_file():
    for root in (_Path("/kaggle/input"), _Path("/kaggle/working"), _Path(".")):
        if not root.exists():
            continue
        for name in ("codes.jsonl.gz", "codes.jsonl", "codes.jsonl.gz.tmp"):
            for path in root.rglob(name):
                if path.is_file():
                    return path
    return None


def _open_any(path):
    with open(path, "rb") as probe:
        packed = probe.read(2) == b"\x1f\x8b"
    return _gzip.open(path, "rt", encoding="utf-8") if packed else open(path, "r", encoding="utf-8")


def code_languages():
    """``code_id -> language`` from the attached clean-data bundle."""
    if _LANGUAGE_CACHE:
        return _LANGUAGE_CACHE
    path = _resolve_codes_file()
    if path is None:
        print("[language-breakdown] codes.jsonl not found; breakdown will be skipped.")
        return _LANGUAGE_CACHE
    with _open_any(path) as stream:
        for line in stream:
            if not line.strip():
                continue
            record = _json.loads(line)
            code_id = str(record.get("code_id", record.get("id", record.get("idx", ""))))
            _LANGUAGE_CACHE[code_id] = str(record.get("language", record.get("lang", "unknown")))
    print(f"[language-breakdown] languages loaded for {len(_LANGUAGE_CACHE):,} codes.")
    return _LANGUAGE_CACHE


def _binary_scores(labels, predicted):
    labels = _np.asarray(labels, dtype=_np.int64)
    predicted = _np.asarray(predicted, dtype=_np.int64)
    tp = int(((predicted == 1) & (labels == 1)).sum())
    fp = int(((predicted == 1) & (labels == 0)).sum())
    tn = int(((predicted == 0) & (labels == 0)).sum())
    fn = int(((predicted == 0) & (labels == 1)).sum())
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    return {
        "P": precision, "R": recall, "F1": f1,
        "Acc": (tp + tn) / max(1, len(labels)),
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "Pairs": int(len(labels)), "Positives": int((labels == 1).sum()),
    }


def record_language_breakdown(frame, scores, threshold, *, dataset, method, graph_type=None):
    """Partition this run's test predictions by pair language and record them."""
    languages = code_languages()
    if not languages or frame is None or not len(frame):
        return []
    scores = _np.asarray(scores, dtype=_np.float64).reshape(-1)
    labels = _np.asarray(frame["label"], dtype=_np.int64).reshape(-1)
    if len(scores) != len(labels):
        print(f"[language-breakdown] skipped {method}: {len(scores)} scores vs {len(labels)} labels.")
        return []
    predicted = (scores >= float(threshold)).astype(_np.int64)

    left = [languages.get(str(value), "unknown") for value in frame["left_id"]]
    right = [languages.get(str(value), "unknown") for value in frame["right_id"]]
    # Cross-language pairs get their own bucket instead of being attributed to
    # one side; ATCoder is entirely java<->python and would otherwise vanish.
    keys = [a if a == b else f"{min(a, b)}->{max(a, b)}" for a, b in zip(left, right)]

    rows = []
    for key in sorted(set(keys)):
        mask = _np.asarray([value == key for value in keys])
        row = {"Dataset": dataset, "Method": method, "GraphType": graph_type or "", "Language": key}
        row.update(_binary_scores(labels[mask], predicted[mask]))
        row["Threshold"] = float(threshold)
        rows.append(row)
    overall = {"Dataset": dataset, "Method": method, "GraphType": graph_type or "", "Language": "ALL"}
    overall.update(_binary_scores(labels, predicted))
    overall["Threshold"] = float(threshold)
    rows.append(overall)

    LANGUAGE_BREAKDOWN_ROWS.extend(rows)
    table = _pd.DataFrame(LANGUAGE_BREAKDOWN_ROWS)
    out_path = _Path("/kaggle/working") / f"{dataset}_language_breakdown.csv"
    try:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        table.to_csv(out_path, index=False)
    except OSError:
        out_path = _Path(f"{dataset}_language_breakdown.csv")
        table.to_csv(out_path, index=False)
    print(f"\n[language-breakdown] {method}{'/' + graph_type if graph_type else ''}")
    print(_pd.DataFrame(rows)[["Language", "P", "R", "F1", "Acc", "Pairs", "Positives"]].to_string(index=False))
    print(f"[language-breakdown] written to {out_path}")
    return rows

DATASET_KEY_FOR_BREAKDOWN = "semanticclonebench_v3"


# Zero-shot cross-language transfer — ASTNN

For each source language (Java, Python, C, C#), train on source-language train/valid pairs only, freeze that source threshold, then test every target language. Attach semanticclonebench_v3_clean_data.zip.


In [ ]:
# Executes the unchanged baseline pipeline once per dataset in isolated state.
# This run writes SemanticCloneBench V3-only CSV files, for example xglue4_*_results.csv.
def run_one_dataset(dataset_key: str, train_language: str | None = None, test_language: str | None = None):
    from pathlib import Path
    import time

    # End-to-end method runtime: loading + preprocessing + training + evaluation.
    run_started = time.perf_counter()

    DATASET_KEY = dataset_key
    KAGGLE_DATA_ROOT = Path("/kaggle/input") / DATASET_KEY
    WORK_DIR = Path("/kaggle/working")
    SEED = 42
    DEVICE = "cuda"
    METHOD_NAME = "ASTNN"

    MAX_TRAIN_PAIRS = RUN_CONFIG["max_train_pairs"]
    MAX_VALID_PAIRS = RUN_CONFIG["max_valid_pairs"]
    MAX_TEST_PAIRS = RUN_CONFIG["max_test_pairs"]

    # Structural AST input exported in graph_spectra.jsonl.gz.
    MAX_AST_NODES = 256
    MAX_AST_EDGES = 512
    MAX_STATEMENTS = 64
    MAX_NODE_TYPES = 20_000

    EPOCHS = RUN_CONFIG["epochs"]
    BATCH_SIZE = 512
    EMBED_DIM = 128
    TREE_HIDDEN_DIM = 192
    CODE_DIM = 192
    DROPOUT = 0.20
    LEARNING_RATE = 5e-4
    WEIGHT_DECAY = 1e-4
    PATIENCE = RUN_CONFIG["patience"]
    USE_AMP = True
    NUM_WORKERS = 0

    RESULTS_PATH = WORK_DIR / f"{DATASET_KEY}_astnn_baseline_results.csv"
    HISTORY_PATH = WORK_DIR / f"{DATASET_KEY}_astnn_baseline_training_history.csv"

    # =========================
    # Clean-data loading and AST preparation
    # =========================
    import gzip
    import json
    import random

    import numpy as np
    import pandas as pd
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    from tqdm.auto import tqdm


    def seed_everything(seed: int = 42):
        random.seed(seed)
        np.random.seed(seed)
        try:
            import torch
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(seed)
        except Exception:
            pass


    def is_gzip_file(path: Path) -> bool:
        try:
            with path.open("rb") as f:
                return f.read(2) == b"\x1f\x8b"
        except OSError:
            return False


    def open_text(path: Path):
        return gzip.open(path, "rt", encoding="utf-8") if is_gzip_file(path) else path.open("r", encoding="utf-8")


    def candidate_roots():
        return [root for root in [KAGGLE_DATA_ROOT, Path("/kaggle/input")] if root.exists()]


    def resolve_file_path(path: Path, *names: str) -> Path:
        if path.is_file():
            return path
        if path.is_dir():
            matches = []
            for name in names:
                matches.extend(item for item in path.rglob(name) if item.is_file())
            if matches:
                return sorted(matches, key=lambda item: (len(item.relative_to(path).parts), len(item.name), str(item)))[0]
        return path


    def find_file(*names: str) -> Path:
        matches = []
        for root in candidate_roots():
            for name in names:
                direct = root / name
                if direct.is_file():
                    matches.append(direct)
                elif direct.is_dir():
                    resolved = resolve_file_path(direct, *names)
                    if resolved.is_file():
                        matches.append(resolved)
                matches.extend(item for item in root.rglob(name) if item.is_file())
        if not matches:
            raise FileNotFoundError(f"Could not find {names} below {[str(root) for root in candidate_roots()]}")
        return sorted(matches, key=lambda item: (len(item.parts), len(item.name), str(item)))[0]


    def load_pairs(path: Path) -> pd.DataFrame:
        path = resolve_file_path(path, "pairs.csv.gz", "pairs.csv", "pairs.csv.gz.tmp")
        compression = "gzip" if is_gzip_file(path) else None
        frame = pd.read_csv(path, compression=compression, dtype={"left_id": str, "right_id": str, "split": str, "label": np.int64})
        frame["left_id"] = frame["left_id"].astype(str)
        frame["right_id"] = frame["right_id"].astype(str)
        return frame


    def maybe_limit_split(frame: pd.DataFrame, split: str, max_rows: int | None, seed: int) -> pd.DataFrame:
        part = frame[frame["split"] == split].copy()
        if max_rows is not None and len(part) > max_rows:
            part = part.sample(n=max_rows, random_state=seed)
        return part.reset_index(drop=True)


    def metric_dict(labels, scores, threshold: float) -> dict:
        pred = (np.asarray(scores) >= threshold).astype(np.int64)
        labels = np.asarray(labels).astype(np.int64)
        p, r, f1, _ = precision_recall_fscore_support(labels, pred, average="binary", zero_division=0)
        acc = accuracy_score(labels, pred)
        tp = int(((pred == 1) & (labels == 1)).sum())
        fp = int(((pred == 1) & (labels == 0)).sum())
        tn = int(((pred == 0) & (labels == 0)).sum())
        fn = int(((pred == 0) & (labels == 1)).sum())
        return {"P": p, "R": r, "F1": f1, "Acc": acc, "TP": tp, "FP": fp, "TN": tn, "FN": fn}


    def choose_threshold(labels, scores, n_grid: int = 401) -> tuple[float, dict]:
        labels = np.asarray(labels).astype(np.int64)
        scores = np.asarray(scores, dtype=np.float32)
        thresholds = np.unique(np.concatenate([np.quantile(scores, np.linspace(0.0, 1.0, n_grid)), np.array([0.5])]))
        best_threshold, best_metrics = 0.5, None
        for threshold in thresholds:
            metrics = metric_dict(labels, scores, float(threshold))
            if best_metrics is None or (metrics["F1"], metrics["Acc"]) > (best_metrics["F1"], best_metrics["Acc"]):
                best_threshold, best_metrics = float(threshold), metrics
        return best_threshold, best_metrics


    def ast_record(row: dict) -> dict | None:
        adjacency = row.get("graphs", {}).get("ast", {}).get("adjacency", {})
        node_types = adjacency.get("node_types")
        # A handful of graph-export failures can lack AST attributes.
        # Ignore only that code record; pair filtering below preserves valid splits.
        if not node_types:
            return None
        node_count = min(int(adjacency.get("num_nodes", 0)), len(node_types), MAX_AST_NODES)
        if node_count <= 0:
            return None
        parents, children = [], []
        for parent, child in zip(adjacency.get("row", []), adjacency.get("col", [])):
            parent, child = int(parent), int(child)
            if parent < node_count and child < node_count and len(parents) < MAX_AST_EDGES:
                parents.append(parent)
                children.append(child)
        child_map = {idx: [] for idx in range(node_count)}
        indegree = [0] * node_count
        for parent, child in zip(parents, children):
            child_map[parent].append(child)
            indegree[child] += 1
        roots = [idx for idx, degree in enumerate(indegree) if degree == 0]
        method_roots = [idx for idx in roots if str(node_types[idx]).upper() == "METHOD"] or roots[:1]
        statement_roots = []
        for method_root in method_roots:
            direct_children = child_map.get(method_root, [])
            block_children = [idx for idx in direct_children if str(node_types[idx]).upper() == "BLOCK"]
            if block_children:
                for block in block_children:
                    statement_roots.extend(child_map.get(block, []))
            else:
                statement_roots.extend(direct_children)
        if not statement_roots:
            statement_roots = method_roots or [0]
        statement_roots = list(dict.fromkeys(statement_roots))[:MAX_STATEMENTS]
        return {"types": [str(value) for value in node_types[:node_count]], "parents": parents, "children": children, "statements": statement_roots}


    def load_ast_records(path: Path, wanted_ids: set[str]) -> dict[str, dict]:
        records = {}
        with open_text(path) as f:
            for line in tqdm(f, desc="Loading AST structures", unit="code"):
                if not line.strip():
                    continue
                row = json.loads(line)
                code_id = str(row.get("code_id"))
                if code_id in wanted_ids:
                    record = ast_record(row)
                    if record is not None:
                        records[code_id] = record
        return records


    seed_everything(SEED)
    pairs_path = find_file("pairs.csv.gz", "pairs.csv", "pairs.csv.gz.tmp")
    graphs_path = find_file("graph_spectra.jsonl.gz", "graph_spectra.jsonl", "graph_spectra.jsonl.gz.tmp")
    pairs_df = load_pairs(pairs_path)

    # --- zero-shot cross-language split -------------------------------------
    language_path = find_file("codes.jsonl.gz", "codes.jsonl", "codes.jsonl.gz.tmp")
    code_languages = {}
    with open_text(language_path) as _language_handle:
        for _language_line in _language_handle:
            if _language_line.strip():
                _language_row = json.loads(_language_line)
                code_languages[str(_language_row.get("code_id"))] = str(_language_row.get("language", "")).lower()

    def _mono_language(frame: pd.DataFrame, language: str) -> pd.DataFrame:
        left = frame["left_id"].astype(str).map(code_languages)
        right = frame["right_id"].astype(str).map(code_languages)
        return frame[(left == language) & (right == language)].reset_index(drop=True)
    wanted_ids = set(pairs_df.left_id) | set(pairs_df.right_id)
    ast_by_id = load_ast_records(graphs_path, wanted_ids)
    print(f"AST coverage: {len(ast_by_id):,}/{len(wanted_ids):,} code records")
    if not ast_by_id:
        raise RuntimeError("No usable AST records were loaded; verify graph_spectra attachment.")
    pairs_df = pairs_df[pairs_df.left_id.isin(ast_by_id) & pairs_df.right_id.isin(ast_by_id)].reset_index(drop=True)

    train_df = maybe_limit_split(pairs_df, "train", MAX_TRAIN_PAIRS, SEED)
    valid_df = maybe_limit_split(pairs_df, "valid", MAX_VALID_PAIRS, SEED + 1)
    test_df = maybe_limit_split(pairs_df, "test", MAX_TEST_PAIRS, SEED + 2)
    print("usable pairs:")
    print(pairs_df.groupby(["split", "label"]).size())
    print(f"using train/valid/test={len(train_df):,}/{len(valid_df):,}/{len(test_df):,}")

    if train_language is not None:
        train_df = _mono_language(train_df, train_language)
        valid_df = _mono_language(valid_df, train_language)
    if test_language is not None:
        test_df = _mono_language(test_df, test_language)
    if train_df.empty or valid_df.empty or test_df.empty:
        raise RuntimeError(
            f"Empty language split: train={train_language!r}, test={test_language!r}; "
            f"counts={len(train_df)}/{len(valid_df)}/{len(test_df)}"
        )
    print(f"cross-language train={train_language}, test={test_language}: "
          f"{len(train_df):,}/{len(valid_df):,}/{len(test_df):,}")

    train_ids = set(train_df.left_id) | set(train_df.right_id)
    type_frequency = {}
    for code_id in train_ids:
        for node_type in ast_by_id[code_id]["types"]:
            type_frequency[node_type] = type_frequency.get(node_type, 0) + 1
    vocab_items = sorted(type_frequency.items(), key=lambda item: (-item[1], item[0]))[: MAX_NODE_TYPES - 2]
    type_vocab = {"<pad>": 0, "<unk>": 1}
    type_vocab.update({node_type: index + 2 for index, (node_type, _) in enumerate(vocab_items)})

    needed_ids = sorted(set(train_df.left_id) | set(train_df.right_id) | set(valid_df.left_id) | set(valid_df.right_id) | set(test_df.left_id) | set(test_df.right_id))
    id_to_row = {code_id: index for index, code_id in enumerate(needed_ids)}
    ast_node_types = np.zeros((len(needed_ids), MAX_AST_NODES), dtype=np.int32)
    ast_parents = np.full((len(needed_ids), MAX_AST_EDGES), -1, dtype=np.int32)
    ast_children = np.full((len(needed_ids), MAX_AST_EDGES), -1, dtype=np.int32)
    ast_statements = np.full((len(needed_ids), MAX_STATEMENTS), -1, dtype=np.int32)
    for row_index, code_id in enumerate(tqdm(needed_ids, desc="Encoding AST structures", unit="code")):
        record = ast_by_id[code_id]
        ast_node_types[row_index, : len(record["types"])] = [type_vocab.get(node_type, 1) for node_type in record["types"]]
        ast_parents[row_index, : len(record["parents"])] = record["parents"]
        ast_children[row_index, : len(record["children"])] = record["children"]
        ast_statements[row_index, : len(record["statements"])] = record["statements"]
    print(f"codes={len(needed_ids):,}; node_types={len(type_vocab):,}; AST tensor={ast_node_types.shape}")


    # =========================
    # ASTNN-style tree encoder and Siamese pair model
    # =========================
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset

    DEVICE = "cuda" if torch.cuda.is_available() and DEVICE == "cuda" else "cpu"
    print("Device:", DEVICE)


    class PairDataset(Dataset):
        def __init__(self, frame: pd.DataFrame):
            self.left = frame["left_id"].map(id_to_row).to_numpy(np.int64)
            self.right = frame["right_id"].map(id_to_row).to_numpy(np.int64)
            self.labels = frame["label"].to_numpy(np.float32)

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, index):
            return (
                torch.tensor(self.left[index], dtype=torch.long),
                torch.tensor(self.right[index], dtype=torch.long),
                torch.tensor(self.labels[index], dtype=torch.float32),
            )


    class ASTNNTreeEncoder(nn.Module):
        """Bottom-up tree convolution followed by a BiGRU over statement subtrees."""
        def __init__(self):
            super().__init__()
            self.embedding = nn.Embedding(len(type_vocab), EMBED_DIM, padding_idx=0)
            self.self_layers = nn.ModuleList([nn.Linear(EMBED_DIM if index == 0 else TREE_HIDDEN_DIM, TREE_HIDDEN_DIM) for index in range(2)])
            self.child_layers = nn.ModuleList([nn.Linear(EMBED_DIM if index == 0 else TREE_HIDDEN_DIM, TREE_HIDDEN_DIM, bias=False) for index in range(2)])
            self.statement_gru = nn.GRU(TREE_HIDDEN_DIM, TREE_HIDDEN_DIM, batch_first=True, bidirectional=True)
            self.output = nn.Sequential(nn.Linear(TREE_HIDDEN_DIM * 2, CODE_DIM), nn.Tanh())

        def forward(self, node_types, parents, children, statements):
            node_mask = node_types.ne(0)
            hidden = self.embedding(node_types)
            valid_edges = parents.ge(0) & children.ge(0)
            for self_layer, child_layer in zip(self.self_layers, self.child_layers):
                child_index = children.clamp_min(0).unsqueeze(-1).expand(-1, -1, hidden.size(-1))
                child_hidden = hidden.gather(1, child_index) * valid_edges.unsqueeze(-1)
                aggregate = torch.zeros_like(hidden)
                parent_index = parents.clamp_min(0).unsqueeze(-1).expand(-1, -1, hidden.size(-1))
                aggregate.scatter_add_(1, parent_index, child_hidden)
                child_count = torch.zeros_like(hidden[..., :1]).scatter_add_(1, parents.clamp_min(0).unsqueeze(-1), valid_edges.unsqueeze(-1).to(hidden.dtype))
                hidden = F.relu(self_layer(hidden) + child_layer(aggregate / child_count.clamp_min(1.0)))
                hidden = hidden * node_mask.unsqueeze(-1)
            statement_mask = statements.ge(0)
            statement_index = statements.clamp_min(0).unsqueeze(-1).expand(-1, -1, hidden.size(-1))
            statement_vectors = hidden.gather(1, statement_index) * statement_mask.unsqueeze(-1)
            sequence, _ = self.statement_gru(statement_vectors)
            pooled = (sequence * statement_mask.unsqueeze(-1)).sum(1) / statement_mask.sum(1).clamp_min(1).unsqueeze(-1)
            return self.output(pooled)


    class SiameseASTNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.register_buffer("node_types", torch.from_numpy(ast_node_types), persistent=False)
            self.register_buffer("parents", torch.from_numpy(ast_parents), persistent=False)
            self.register_buffer("child_indices", torch.from_numpy(ast_children), persistent=False)
            self.register_buffer("statements", torch.from_numpy(ast_statements), persistent=False)
            self.encoder = ASTNNTreeEncoder()
            self.classifier = nn.Sequential(
                nn.Linear(CODE_DIM * 4, TREE_HIDDEN_DIM * 2),
                nn.GELU(),
                nn.Dropout(DROPOUT),
                nn.Linear(TREE_HIDDEN_DIM * 2, TREE_HIDDEN_DIM),
                nn.GELU(),
                nn.Dropout(DROPOUT),
                nn.Linear(TREE_HIDDEN_DIM, 1),
            )

        def encode(self, code_indices):
            return self.encoder(
                self.node_types[code_indices].long(),
                self.parents[code_indices].long(),
                self.child_indices[code_indices].long(),
                self.statements[code_indices].long(),
            )

        def forward(self, left_indices, right_indices):
            left = self.encode(left_indices)
            right = self.encode(right_indices)
            features = torch.cat([left, right, torch.abs(left - right), left * right], dim=1)
            return self.classifier(features).squeeze(1)


    def make_loader(frame: pd.DataFrame, shuffle: bool) -> DataLoader:
        return DataLoader(
            PairDataset(frame),
            batch_size=BATCH_SIZE,
            shuffle=shuffle,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE == "cuda"),
            drop_last=False,
        )


    @torch.no_grad()
    def predict_scores(model, frame: pd.DataFrame) -> np.ndarray:
        model.eval()
        scores = []
        for left, right, _ in tqdm(make_loader(frame, shuffle=False), desc="Predict", leave=False):
            logits = model(left.to(DEVICE, non_blocking=True), right.to(DEVICE, non_blocking=True))
            scores.append(torch.sigmoid(logits).float().cpu().numpy())
        return np.concatenate(scores) if scores else np.array([], dtype=np.float32)


    model = SiameseASTNN().to(DEVICE)
    trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    print(f"trainable parameters: {trainable_parameters:,}")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE == "cuda"))
    train_loader = make_loader(train_df, shuffle=True)
    history, best_f1, best_state, bad_epochs = [], -1.0, None, 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, total_count = 0.0, 0
        progress = tqdm(train_loader, desc=f"{METHOD_NAME} epoch {epoch:02d}")
        for left, right, labels in progress:
            left = left.to(DEVICE, non_blocking=True)
            right = right.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(USE_AMP and DEVICE == "cuda")):
                loss = criterion(model(left, right), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += float(loss.detach().cpu()) * labels.numel()
            total_count += labels.numel()
            progress.set_postfix(loss=total_loss / max(total_count, 1))

        valid_scores = predict_scores(model, valid_df)
        threshold, valid_metrics = choose_threshold(valid_df["label"].to_numpy(), valid_scores)
        row = {
            "Method": METHOD_NAME,
            "epoch": epoch,
            "train_loss": total_loss / max(total_count, 1),
            "valid_P": valid_metrics["P"],
            "valid_R": valid_metrics["R"],
            "valid_F1": valid_metrics["F1"],
            "valid_Acc": valid_metrics["Acc"],
            "threshold": threshold,
        }
        history.append(row)
        print(row)
        if valid_metrics["F1"] > best_f1:
            best_f1, best_threshold, best_epoch = valid_metrics["F1"], threshold, epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("early stopping")
                break

    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
    model.load_state_dict(best_state)
    test_scores = predict_scores(model, test_df)
    test_metrics = metric_dict(test_df["label"].to_numpy(), test_scores, best_threshold)
    record_language_breakdown(test_df, test_scores, best_threshold, dataset=DATASET_KEY_FOR_BREAKDOWN, method="ASTNN")
    result = {
        "Method": METHOD_NAME,
        "BestEpoch": best_epoch,
        "BestValidF1": best_f1,
        **test_metrics,
        "Threshold": best_threshold,
        "TrainPairs": len(train_df),
        "ValidPairs": len(valid_df),
        "TestPairs": len(test_df),
        "TrainableParameters": trainable_parameters,
    }
    result["TrainableParameters"] = int(sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)) if "model" in locals() else 0
    result["RuntimeSeconds"] = float(time.perf_counter() - run_started)
    result["RuntimeMinutes"] = result["RuntimeSeconds"] / 60.0
    pd.DataFrame([result]).to_csv(RESULTS_PATH, index=False)
    print(pd.DataFrame([result]))
    print("saved:", RESULTS_PATH)

    # Research-reproducibility manifest and enriched result table.
    # This is deliberately written after evaluation so measured runtime is final.
    import datetime as _datetime
    try:
        _gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
        _gpu_capability = ".".join(map(str, torch.cuda.get_device_capability(0))) if torch.cuda.is_available() else None
        _torch_version = torch.__version__
    except Exception:
        _gpu_name, _gpu_capability, _torch_version = "unavailable", None, "unavailable"
    _completed_utc = _datetime.datetime.now(_datetime.timezone.utc).isoformat()
    _run_seconds = float(time.perf_counter() - run_started)
    _shared_fields = {
        "Dataset": DATASET_KEY,
        "RunProfile": RUN_PROFILE,
        "Seed": int(SEED),
        "ConfiguredEpochs": int(EPOCHS) if "EPOCHS" in locals() else None,
        "BatchSize": int(BATCH_SIZE) if "BATCH_SIZE" in locals() else None,
        "LearningRate": float(LEARNING_RATE) if "LEARNING_RATE" in locals() else None,
        "WeightDecay": float(WEIGHT_DECAY) if "WEIGHT_DECAY" in locals() else None,
        "GPU": _gpu_name,
        "GPUCapability": _gpu_capability,
        "TorchVersion": _torch_version,
        "CompletedUTC": _completed_utc,
    }
    if "results_df" in locals():
        _result_table = results_df.copy()
    elif "result" in locals():
        _result_table = pd.DataFrame([result])
    elif "row" in locals():
        _result_table = pd.DataFrame([row])
    else:
        _result_table = pd.DataFrame()
    for _field, _value in _shared_fields.items():
        _result_table[_field] = _value
    if "RuntimeSeconds" not in _result_table.columns:
        _result_table["RuntimeSeconds"] = _run_seconds
    if "RuntimeMinutes" not in _result_table.columns:
        _result_table["RuntimeMinutes"] = _run_seconds / 60.0
    _result_path = RESULTS_PATH if "RESULTS_PATH" in locals() else out_path
    _result_table.to_csv(_result_path, index=False)
    _metadata = {
        **_shared_fields,
        "RunLabel": RUN_LABEL,
        "RuntimeSeconds": _run_seconds,
        "RuntimeMinutes": _run_seconds / 60.0,
        "RequestedPairCaps": {
            "train": MAX_TRAIN_PAIRS,
            "valid": MAX_VALID_PAIRS,
            "test": MAX_TEST_PAIRS,
        },
        "ModelConfiguration": {
            _name: locals().get(_name)
            for _name in (
                "MAX_AST_NODES", "MAX_AST_EDGES", "MAX_STATEMENTS", "MAX_NODE_TYPES", "MAX_NODES",
                "EMBED_DIM", "HIDDEN_DIM", "TREE_HIDDEN_DIM", "CODE_DIM", "DROPOUT",
                "GRAPH_TYPE", "GRAPH_TYPES", "K_EIGEN", "USE_EIGEN_STATS", "USE_GRAPH_STATS",
            ) if _name in locals()
        },
        "OutputFiles": {
            "results": str(_result_path),
            "history": str(HISTORY_PATH) if "HISTORY_PATH" in locals() else (str(history_path) if "history_path" in locals() else None),
        },
    }
    _metadata_path = WORK_DIR / f"{DATASET_KEY}_{RUN_LABEL}_run_metadata.json"
    _metadata_path.write_text(json.dumps(_metadata, indent=2, default=str), encoding="utf-8")
    print("Research metadata:", _metadata_path)
    if "results_df" in locals():
        results_df = _result_table

    if "results_df" in locals():
        return results_df.copy()
    if "result" in locals():
        return pd.DataFrame([result])
    if "row" in locals():
        return pd.DataFrame([row])
    raise RuntimeError("The baseline did not produce a result table.")


DISPLAY_METHOD = "ASTNN"

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

BASE_DATASET_KEY = DATASET_KEYS[0]
CROSS_LANGUAGE_SOURCES = ["java", "python", "c", "csharp"]
all_language_rows = []
for source_language in CROSS_LANGUAGE_SOURCES:
    for target_language in CROSS_LANGUAGE_SOURCES:
        print("\n" + "=" * 96)
        print(f"{DISPLAY_METHOD}: train {source_language} -> test {target_language}")
        print("=" * 96)
        result = run_one_dataset(BASE_DATASET_KEY, source_language, target_language)
        result.insert(0, "Experiment", "cross_language_transfer")
        result.insert(1, "TrainedOnLanguage", source_language)
        result.insert(2, "TestLanguage", target_language)
        all_language_rows.append(result)

cross_language_results = pd.concat(all_language_rows, ignore_index=True)
result_path = Path("/kaggle/working") / f"{RUN_LABEL}_semanticclonebench_v3_cross_language_results.csv"
cross_language_results.to_csv(result_path, index=False)
display(cross_language_results)

matrix = cross_language_results.pivot(index="TrainedOnLanguage", columns="TestLanguage", values="F1")
matrix = matrix.reindex(index=CROSS_LANGUAGE_SOURCES, columns=CROSS_LANGUAGE_SOURCES)
fig, axis = plt.subplots(figsize=(7, 5))
image = axis.imshow(matrix, vmin=0, vmax=1, cmap="viridis")
axis.set_xticks(range(len(CROSS_LANGUAGE_SOURCES)), CROSS_LANGUAGE_SOURCES)
axis.set_yticks(range(len(CROSS_LANGUAGE_SOURCES)), CROSS_LANGUAGE_SOURCES)
axis.set(xlabel="Test language", ylabel="Training language", title=f"Zero-shot transfer F1 — {DISPLAY_METHOD}")
for row in range(len(CROSS_LANGUAGE_SOURCES)):
    for column in range(len(CROSS_LANGUAGE_SOURCES)):
        value = matrix.iloc[row, column]
        axis.text(column, row, "-" if pd.isna(value) else f"{value:.3f}",
                  ha="center", va="center", color="white" if pd.notna(value) and value < .55 else "black", fontsize=9)
fig.colorbar(image, ax=axis, label="Test F1")
fig.tight_layout()
figure_path = Path("/kaggle/working") / f"{RUN_LABEL}_semanticclonebench_v3_cross_language_f1.png"
fig.savefig(figure_path, dpi=180)
plt.show()
print("Saved:", result_path)
print("Saved:", figure_path)
